# 1. Úkoly
## 1.1 `mpi4py`
Vytvořte program v Pythonu (`mpi4py`), který:

1. Spočítá součet čísel `1..n`.
2. Rozdělí výpočet mezi `m` MPI (Message Passing Interface) procesů.
3. Každý proces spočítá součet své části rozsahu.
4. Rank 0 sesbírá dílčí výsledky a vypíše finální součet.

### 1.1.1 Referenční řešení
Jednoduchá varianta pro ověření, že `mpi4py` v prostředí funguje.

In [1]:
%%writefile mpi_sum.py
from mpi4py import MPI
import sys

def main():
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 100
    
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    
    chunk = n // size
    remainder = n % size
    
    start = rank * chunk + min(rank, remainder) + 1
    end = start + chunk + (1 if rank < remainder else 0) - 1
    
    local_sum = sum(range(start, end + 1))
    
    total_sum = comm.reduce(local_sum, op=MPI.SUM, root=0)
    
    if rank == 0:
        print(f"Celkovy soucet cisel 1 az {n} je: {total_sum}")

if __name__ == "__main__":
    main()

Overwriting mpi_sum.py


In [2]:
!mpiexec -n 4 python mpi_sum.py 100

--------------------------------------------------------------------------
mpiexec was unable to find the specified executable file, and therefore
did not launch the job.  This error was first reported for process
rank 0; it may have occurred for other processes as well.

NOTE: A common cause for this error is misspelling a mpiexec command
      line parameter option (remember that mpiexec interprets the first
      unrecognized command line token as the executable).

Node:       DESKTOP-JSO04T9
Executable: python
--------------------------------------------------------------------------
4 total processes failed to start


## 1.2 Linkování C do Pythonu
V souboru `pruchod_grafem.c` je implementovaná funkce:

```c
size_t reachable_in_n_steps(const int32_t *edges, size_t m, size_t n_steps, int32_t *reachable_vertices, size_t reachable_capacity);
```

Úkol:

1. Zkompilujte `pruchod_grafem.c` do dynamické knihovny.
2. Načtěte knihovnu přes `ctypes` a zavolejte `reachable_in_n_steps`.
3. Načtěte knihovnu přes `cffi` a zavolejte `reachable_in_n_steps`.
4. Vytvořte Cython wrapper pro stejnou signaturu.
5. Ověřte výsledky proti očekávanému vektoru.

ABI (Application Binary Interface) a mapování typů:

- `int32_t` <-> `np.int32` / `ctypes.c_int32`.
- `size_t` <-> `ctypes.c_size_t` (v Pythonu nezáporné `int`).
- `edges` je C-contiguous pole délky `2*m` (`[from_0..from_m-1, to_0..to_m-1]`).
- `reachable_vertices` je výstupní C-contiguous pole `np.int32`.
- Funkce vrací počet dosažitelných vrcholů.

### 1.2.0 Prerekvizity
Nejdřív si připravíme vše společné:

1. Hlavička `pruchod_grafem.h` je už přímo ve složce `Week_13` vedle `pruchod_grafem.c`.
2. Složitější testovací data + očekávaný výstup pro validaci.

In [3]:
import numpy as np

# Neorientovaný graf, uložený jako [from_0..from_m-1, to_0..to_m-1]
# Hrany: (0-1), (0-2), (1-3), (2-3), (3-4), (4-5), (2-6), (6-7), (7-8), (5-8), (8-9), (1-6)
EDGES = np.ascontiguousarray([
    0, 0, 1, 2, 3, 4, 2, 6, 7, 5, 8, 1,
    1, 2, 3, 3, 4, 5, 6, 7, 8, 8, 9, 6,
], dtype=np.int32)

M = EDGES.size // 2
N_STEPS = 3
CAPACITY = 10  # vrcholy 0..9
EXPECTED_REACHABLE = np.array([0, 1, 2, 3, 4, 6, 7], dtype=np.int32)

print("m:", M)
print("n_steps:", N_STEPS)
print("expected (<= 3 kroky):", EXPECTED_REACHABLE.tolist())


def validate_solution(name, count, out_vector):
    got = out_vector[:count]
    print(f"{name}: {got.tolist()}")
    assert np.array_equal(got, EXPECTED_REACHABLE), (
        f"{name} vraci jiny vysledek."
        f"\nexpected={EXPECTED_REACHABLE.tolist()}"
        f"\ngot={got.tolist()}"
    )

m: 12
n_steps: 3
expected (<= 3 kroky): [0, 1, 2, 3, 4, 6, 7]


### 1.2.1 Kompilace C souboru do sdílené knihovny

In [4]:
!gcc -shared -fPIC -O3 pruchod_grafem.c -o libpruchod.so

### 1.2.2 Řešení: `ctypes`

In [5]:
import ctypes
import numpy as np

lib_ctypes = ctypes.CDLL('./libpruchod.so')

lib_ctypes.reachable_in_n_steps.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.int32, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_size_t,
    ctypes.c_size_t,
    np.ctypeslib.ndpointer(dtype=np.int32, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_size_t
]
lib_ctypes.reachable_in_n_steps.restype = ctypes.c_size_t

out_vector_ctypes = np.zeros(CAPACITY, dtype=np.int32)

count_ctypes = lib_ctypes.reachable_in_n_steps(EDGES, M, N_STEPS, out_vector_ctypes, CAPACITY)

validate_solution("ctypes", count_ctypes, out_vector_ctypes)

ctypes: [0, 1, 2, 3, 4, 6, 7]


### 1.2.3 Řešení: `cffi` (signatura načtená z `.h`)

In [6]:
from cffi import FFI
import numpy as np

ffi = FFI()

ffi.cdef("""
size_t reachable_in_n_steps(
    const int32_t *edges,
    size_t m,
    size_t n_steps,
    int32_t *reachable_vertices,
    size_t reachable_capacity
);
""")

lib_cffi = ffi.dlopen('./libpruchod.so')

out_vector_cffi = np.zeros(CAPACITY, dtype=np.int32)

edges_ptr = ffi.cast("int32_t *", ffi.from_buffer(EDGES))
out_ptr = ffi.cast("int32_t *", ffi.from_buffer(out_vector_cffi))

count_cffi = lib_cffi.reachable_in_n_steps(edges_ptr, M, N_STEPS, out_ptr, CAPACITY)

validate_solution("cffi", count_cffi, out_vector_cffi)

cffi: [0, 1, 2, 3, 4, 6, 7]


### 1.2.4 Řešení: Cython

In [12]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [13]:
%%cython -I .
# distutils: sources = pruchod_grafem.c
import numpy as np
cimport numpy as cnp
from libc.stdint cimport int32_t

cdef extern from "pruchod_grafem.h":
    size_t reachable_in_n_steps(const int32_t *edges, size_t m, size_t n_steps, int32_t *reachable_vertices, size_t reachable_capacity)

def cython_reachable_wrapper(cnp.ndarray[int32_t, ndim=1, mode="c"] edges, size_t m, size_t n_steps, size_t capacity):
    cdef cnp.ndarray[int32_t, ndim=1, mode="c"] out_vector = np.zeros(capacity, dtype=np.int32)
    cdef size_t count
    
    count = reachable_in_n_steps(&edges[0], m, n_steps, &out_vector[0], capacity)
    return count, out_vector

In [14]:
count_cython, out_vector_cython = cython_reachable_wrapper(EDGES, M, N_STEPS, CAPACITY)
validate_solution("Cython", count_cython, out_vector_cython)

Cython: [0, 1, 2, 3, 4, 6, 7]
